In [28]:

# ── تثبيت المكتبات ──
!pip install opencv-python-headless matplotlib numpy tensorflow pillow \
  tqdm scikit-learn seaborn openai-whisper -q

# ── إخفاء التحذيرات  ─
import warnings
warnings.filterwarnings('ignore')

# ── توصيل Kaggle ──
import os
from google.colab import files


print("⬆️ ارفع ملف kaggle.json من حسابك في Kaggle")
files.upload()

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✅ Kaggle متصل بنجاح!")

⬆️ ارفع ملف kaggle.json من حسابك في Kaggle


Saving kaggle.json to kaggle.json
✅ Kaggle متصل بنجاح!


In [1]:

# 1.1 استيراد المكتبات الضرورية
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
import os

print("✅ تم استيراد المكتبات الأساسية بنجاح.")

✅ تم استيراد المكتبات الأساسية بنجاح.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
model_path = '/content/drive/MyDrive/WLASL_Project/wlasl_model_100words_FINAL.h5'

# 1.3 تحميل النموذج الأفضل
try:
    sign_language_model = load_model(model_path)
    print(f"✅ تم تحميل النموذج '{model_path}' بنجاح!")
    sign_language_model.summary()
except Exception as e:
    print(f"❌ فشل تحميل النموذج من المسار '{model_path}'. يرجى التأكد من وجود الملف وصحة المسار.")
    print(f"الخطأ: {e}")
    sign_language_model = None # تعيين النموذج إلى None في حالة الفشل


✅ تم تحميل النموذج '/content/drive/MyDrive/WLASL_Project/wlasl_model_100words_FINAL.h5' بنجاح!


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 64, 64,    │        648 │ input_layer[0][0] │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 64, 64,    │         96 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 64, 64,    │          0 │ bn_Conv1[0][0]    │
│                     │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │        216 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │         96 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 24)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │        384 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 64, 64,    │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 64, 64,    │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 64, 64,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 65, 65,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 32, 32,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 32, 32,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 32, 32,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 32, 32,    │      2,304 │ block_1_depthwis

 Total params: 1,735,702 (6.62 MB)

 Trainable params: 353,636 (1.35 MB)

 Non-trainable params: 1,382,064 (5.27 MB)

 Optimizer params: 2 (12.00 B)

In [18]:
# 2.1 بناء القاموس من WLASL_v0.3.json
import json
import numpy as np

wlasl_json_path = '/content/drive/MyDrive/wlasl_data/WLASL_v0.3.json'

# تحميل بيانات WLASL
with open(wlasl_json_path, 'r') as f:
    wlasl_data = json.load(f)

# استخراج أسماء الكلمات
all_glosses = [entry['gloss'] for entry in wlasl_data]
print(f"📋 إجمالي الكلمات في WLASL: {len(all_glosses)}")

# أول 100 كلمة (التي دربت عليها الأرجح)
word_list_100 = all_glosses[:100]

# بناء القواميس
index_to_word = {i: word for i, word in enumerate(word_list_100)}
word_to_index = {word: i for i, word in enumerate(word_list_100)}

print(f"\n✅ تم بناء القاموس: {len(index_to_word)} كلمة")
print(f"\n📝 أول 20 كلمة:")
for i in range(100):
    print(f"   {i:3d} → {index_to_word[i]}")

📋 إجمالي الكلمات في WLASL: 2000

✅ تم بناء القاموس: 100 كلمة

📝 أول 20 كلمة:
     0 → book
     1 → drink
     2 → computer
     3 → before
     4 → chair
     5 → go
     6 → clothes
     7 → who
     8 → candy
     9 → cousin
    10 → deaf
    11 → fine
    12 → help
    13 → no
    14 → thin
    15 → walk
    16 → year
    17 → yes
    18 → all
    19 → black
    20 → cool
    21 → finish
    22 → hot
    23 → like
    24 → many
    25 → mother
    26 → now
    27 → orange
    28 → table
    29 → thanksgiving
    30 → what
    31 → woman
    32 → bed
    33 → blue
    34 → bowling
    35 → can
    36 → dog
    37 → family
    38 → fish
    39 → graduate
    40 → hat
    41 → hearing
    42 → kiss
    43 → language
    44 → later
    45 → man
    46 → shirt
    47 → study
    48 → tall
    49 → white
    50 → wrong
    51 → accident
    52 → apple
    53 → bird
    54 → change
    55 → color
    56 → corn
    57 → cow
    58 → dance
    59 → dark
    60 → doctor
    61 → eat
    62 →

In [11]:
# 3.1 التحقق: هل y_wlasl_100.npy يحتوي على الفئات 0-99؟
y = np.load('/content/drive/MyDrive/WLASL_Project/y_wlasl_100.npy')

print(f"📊 شكل y: {y.shape}")
print(f"📊 القيم الفريدة في y: {len(np.unique(y))} فئة")
print(f"📊 نطاق القيم: {min(y)} → {max(y)}")

# إذا كان عدد الفئات = 100 والنطاق 0-99، فالقاموس صحيح
if len(np.unique(y)) == 100 and min(y) == 0 and max(y) == 99:
    print("✅ القاموس متوافق مع بيانات التدريب!")
else:
    print("⚠️ قد يكون هناك عدم تطابق - نحتاج مراجعة إضافية")

📊 شكل y: (15000,)
📊 القيم الفريدة في y: 100 فئة
📊 نطاق القيم: 0 → 99
✅ القاموس متوافق مع بيانات التدريب!


In [12]:
# 4.1 حفظ القاموس كملف JSON
class_indices_path = '/content/drive/MyDrive/WLASL_Project/class_indices.json'

# حفظ بنفس الصيغة التي يستخدمها Keras
class_indices = {word: idx for word, idx in word_to_index.items()}

with open(class_indices_path, 'w') as f:
    json.dump(class_indices, f, indent=2)

print(f"✅ تم حفظ القاموس في: {class_indices_path}")

# 4.2 حفظ القاموس العكسي أيضاً
index_to_word_path = '/content/drive/MyDrive/WLASL_Project/index_to_word.json'
with open(index_to_word_path, 'w') as f:
    # تحويل المفاتيح إلى نص لأن JSON لا يدعم أرقام كمفاتيح
    json.dump({str(k): v for k, v in index_to_word.items()}, f, indent=2)

print(f"✅ تم حفظ القاموس العكسي في: {index_to_word_path}")

✅ تم حفظ القاموس في: /content/drive/MyDrive/WLASL_Project/class_indices.json
✅ تم حفظ القاموس العكسي في: /content/drive/MyDrive/WLASL_Project/index_to_word.json


In [16]:
# حفظ class_indices.json
class_indices_path = '/content/drive/MyDrive/WLASL_Project/class_indices.json'
with open(class_indices_path, 'w') as f:
    json.dump(word_to_index, f, indent=2)
print(f"💾 تم حفظ: {class_indices_path}")

# حفظ index_to_word.json
index_to_word_path = '/content/drive/MyDrive/WLASL_Project/index_to_word.json'
with open(index_to_word_path, 'w') as f:
    json.dump({str(k): v for k, v in index_to_word.items()}, f, indent=2)
print(f"💾 تم حفظ: {index_to_word_path}")

💾 تم حفظ: /content/drive/MyDrive/WLASL_Project/class_indices.json
💾 تم حفظ: /content/drive/MyDrive/WLASL_Project/index_to_word.json


In [19]:
# 3.1 تثبيت Whisper
!pip install -q openai-whisper

# 3.2 استيراد المكتبات
import whisper
import torch

# 3.3 تحميل نموذج Whisper
print("⏳ جاري تحميل نموذج Whisper (قد يستغرق دقيقة)...")
whisper_model = whisper.load_model("small")  # small = توازن جيد
print("✅ تم تحميل نموذج Whisper بنجاح!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 6.0 MB/s eta 0:00:00
⏳ جاري تحميل نموذج Whisper (قد يستغرق دقيقة)...


100%|███████████████████████████████████████| 461M/461M [00:07<00:00, 67.0MiB/s]


✅ تم تحميل نموذج Whisper بنجاح!


In [22]:
def speech_to_text(audio_path, language='en'):
    """
    تحويل ملف صوتي إلى نص

    المعاملات:
        audio_path: مسار ملف الصوت (wav, mp3, m4a, ...)
        language: رمز اللغة ('en' للإنجليزية)

    Returns:
        النص المستخرج من الصوت
    """
    result = whisper_model.transcribe(audio_path, language=language)
    text = result['text'].strip()
    return text
    # الخيار ب: نص تجريبي مباشر (بدون ملف صوتي)
transcribed_text = "I want to eat pizza and drink water"
print(f"📝 نص تجريبي: '{transcribed_text}'")

import re
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import word_tokenize

# تحميل القاموس
import json
with open('/content/drive/MyDrive/WLASL_Project/class_indices.json', 'r') as f:
    word_to_index = json.load(f)

known_words = set(word_to_index.keys())

def process_text(text, known_words):
    """
    معالجة النص وتقسيمه إلى كلمات

    Returns:
        قائمة من القواميس لكل كلمة
    """
    text = text.lower().strip()
    text = re.sub(r'[^\w\s]', '', text)
    words = word_tokenize(text)

    result = []
    for word in words:
        if word in known_words:
            result.append({
                'word': word,
                'type': 'known',
                'index': word_to_index[word]
            })
        else:
            result.append({
                'word': word,
                'type': 'unknown',
                'letters': list(word.upper())
            })
    return result

# اختبار معالجة النص
processed = process_text(transcribed_text, known_words)

print(f"📝 الجملة: '{transcribed_text}'")
print(f"📋 النتيجة:\n")

known_count = 0
unknown_count = 0

for item in processed:
    if item['type'] == 'known':
        print(f"   ✅ '{item['word']}' → إشارة معروفة (index: {item['index']})")
        known_count += 1
    else:
        print(f"   ❓ '{item['word']}' → غير معروفة، تهجئة: {'-'.join(item['letters'])}")
        unknown_count += 1

print(f"\n📊 الإحصائيات:")
print(f"   كلمات معروفة (لها إشارة): {known_count}")
print(f"   كلمات غير معروفة (تهجئة): {unknown_count}")
print(f"   نسبة التغطية: {known_count/len(processed)*100:.1f}%")

📝 نص تجريبي: 'I want to eat pizza and drink water'
📝 الجملة: 'I want to eat pizza and drink water'
📋 النتيجة:

   ❓ 'i' → غير معروفة، تهجئة: I
   ✅ 'want' → إشارة معروفة (index: 74)
   ❓ 'to' → غير معروفة، تهجئة: T-O
   ✅ 'eat' → إشارة معروفة (index: 61)
   ✅ 'pizza' → إشارة معروفة (index: 68)
   ❓ 'and' → غير معروفة، تهجئة: A-N-D
   ✅ 'drink' → إشارة معروفة (index: 1)
   ❓ 'water' → غير معروفة، تهجئة: W-A-T-E-R

📊 الإحصائيات:
   كلمات معروفة (لها إشارة): 4
   كلمات غير معروفة (تهجئة): 4
   نسبة التغطية: 50.0%


In [23]:
# ========================================
#  Load Whisper + Functions
# ========================================
import os, json, difflib, cv2, shutil, numpy as np
from base64 import b64encode
from IPython.display import HTML, display

# ── تثبيت faster-whisper (أسرع وأخف 4 مرات) ──
try:
    from faster_whisper import WhisperModel
    USE_FASTER = True
except ImportError:
    print("Installing faster-whisper...")
    !pip install -q faster-whisper
    from faster_whisper import WhisperModel
    USE_FASTER = True

print(f"✅ Using faster-whisper" if USE_FASTER else "⚠️ Using openai-whisper")

# ── تحميل Whisper ──
print("Loading Whisper (base)...")
if USE_FASTER:
    whisper_model = WhisperModel("base", device="auto", compute_type="int8")
else:
    import whisper
    whisper_model = whisper.load_model("base")
print("✅ Whisper loaded!")

# ── تحميل النموذج والتسميات ──
MODEL_PATH = '/content/drive/MyDrive/wlasl_data/wlasl_model_100words_FINAL.h5'
LABELS_PATH = '/content/drive/MyDrive/wlasl_data/class_indices.json'

has_model = False
idx_to_word = {}
available_words = []

if os.path.exists(MODEL_PATH):
    from tensorflow.keras.models import load_model
    model = load_model(MODEL_PATH)
    has_model = True
    with open(LABELS_PATH, 'r') as f:
        labels = json.load(f)
    idx_to_word = {int(k): v for k, v in labels.items()}
    available_words = list(idx_to_word.values())
    print(f"✅ Model loaded! ({len(available_words)} words)")
else:
    print(f"⚠️ Model not found at {MODEL_PATH}")
    print("   Running in TEXT-MATCH mode only (no video classification)")

# ── تحميل بيانات WLASL ──
wlasl_data = None
for path in ['/content/WLASL_v0.3.json', '/content/WLASL/info.json']:
    if os.path.exists(path):
        with open(path, 'r') as f:
            wlasl_data = json.load(f)
        print(f"✅ WLASL data loaded from {path}")
        break

if wlasl_data is None:
    raise FileNotFoundError("❌ WLASL data not found!")

# ── مجلد فيديوهات الإشارة ──
SIGN_VIDEO_DIR = '/content/drive/MyDrive/WLASL_Project/sign_videos'
os.makedirs(SIGN_VIDEO_DIR, exist_ok=True)

# ── كلمات الربط الشائعة (ستُتجاهل) ──
STOP_WORDS = {
    'a', 'an', 'the', 'is', 'am', 'are', 'was', 'were', 'be',
    'to', 'of', 'in', 'for', 'on', 'at', 'by', 'with', 'from',
    'and', 'or', 'if', 'so', 'not', 'do',
    'my', 'your', 'his', 'her', 'its', 'our', 'their',
    'this', 'that', 'it', 'i', 'me', 'you', 'he', 'she',
    'we', 'they', 'have', 'has', 'had', 'will', 'would',
    'could', 'should', 'may', 'might', 'shall'
}

# ── بناء lookup سريع ──
word_video_cache = {}
for entry in wlasl_data:
    word = entry['gloss'].lower()
    if word not in word_video_cache and entry['instances']:
        vid_id = entry['instances'][0]['video_id']
        word_video_cache[word] = vid_id

# ── تحقق من ffmpeg ──
ffmpeg_available = os.system('which ffmpeg > /dev/null 2>&1') == 0
if not ffmpeg_available:
    print("⚠️ ffmpeg not found. Installing...")
    !apt-get install -q -y ffmpeg > /dev/null 2>&1

# ══════════════════════════════════════════════════════════════
#  🤟 دالة: تشغيل فيديو الإشارة
# ══════════════════════════════════════════════════════════════
def get_sign_video(word, max_width=300):
    """تعرض فيديو إشارة لكلمة معينة"""
    vid_id = word_video_cache.get(word.lower())
    if not vid_id:
        print(f"  ❌ '{word}' not in video cache")
        return None

    src_path = f'/content/videos/{vid_id}.mp4'
    if not os.path.exists(src_path):
        print(f"  ❌ Video file '{vid_id}.mp4' not found")
        return None

    # تحويل الفيديو
    web_path = f'{SIGN_VIDEO_DIR}/{word.lower()}.mp4'
    if not os.path.exists(web_path):
        ret = os.system(
            f'ffmpeg -y -i {src_path} '
            f'-c:v libx264 -preset fast -crf 23 -an '
            f'-vf "scale={max_width}:-2" {web_path} '
            f'> /dev/null 2>&1'
        )
        if ret != 0:
            print(f"  ❌ ffmpeg failed for '{word}'")
            return None

    # عرض الفيديو
    try:
        with open(web_path, 'rb') as f:
            mp4 = b64encode(f.read()).decode()
        display(HTML(
            f'<div style="margin:5px 0;">'
            f'<b style="color:#2196F3;">🤟 {word.upper()}</b><br>'
            f'<video width="{max_width}" controls>'
            f'<source src="data:video/mp4;base64,{mp4}" type="video/mp4">'
            f'</video></div>'
        ))
        return web_path
    except Exception as e:
        print(f"  ❌ Display error: {e}")
        return None


# ══════════════════════════════════════════════════════════════
#  🎤 دالة رئيسية: صوت → نص → إشارة
# ══════════════════════════════════════════════════════════════
def speech_to_sign_final(audio_path, similarity_threshold=0.7):
    """
    تحوّل صوت مسجل → نص → فيديوهات إشارة
    """
    print("=" * 60)
    print("  🎤 Speech → Text → Sign Language Video")
    print("=" * 60)

    # ── الخطوة 1: تحويل الصوت لنص ──
    print("\n[1/3] Transcribing audio...")
    try:
        if USE_FASTER:
            segments, info = whisper_model.transcribe(
                audio_path, language="en",
                beam_size=5, vad_filter=True  # ← vad يزيل الصمت
            )
            heard = " ".join(seg.text for seg in segments).strip().lower()
        else:
            result = whisper_model.transcribe(audio_path, language="en")
            heard = result["text"].strip().lower()
    except Exception as e:
        print(f"  ❌ Transcription failed: {e}")
        return []

    if not heard:
        print("  ❌ No speech detected!")
        return []

    print(f"   Heard: \"{heard}\"")

    # ── الخطوة 2: تنظيف ومطابقة الكلمات ──
    print("\n[2/3] Matching words...")
    all_words = heard.split()
    found_words = []
    skipped = []
    not_found = []

    for word in all_words:
        # تنظيف
        clean = word.strip(".,!?;:'\"()[]").lower()
        if not clean or len(clean) <= 1:
            continue

        # تجاهل كلمات الربط
        if clean in STOP_WORDS:
            skipped.append(clean)
            continue

        # بحث دقيق
        if clean in available_words:
            found_words.append(clean)
            print(f"   ✅ '{clean}' → exact match")
            continue

        # بحث بالتشابه
        matches = difflib.get_close_matches(
            clean, available_words, n=1,
            cutoff=similarity_threshold
        )
        if matches:
            similarity = difflib.SequenceMatcher(
                None, clean, matches[0]
            ).ratio()
            found_words.append(matches[0])
            print(f"   🔀 '{clean}' → '{matches[0]}' ({similarity:.0%} similar)")
        else:
            not_found.append(clean)
            print(f"   ❌ '{clean}' — no match")

    # ── الخطوة 3: عرض فيديوهات الإشارة ──
    print(f"\n[3/3] Playing {len(found_words)} sign video(s)...\n")

    for word in found_words:
        get_sign_video(word, max_width=280)
        print()

    # ── التقرير النهائي ──
    print("=" * 60)
    print("  📊 Summary Report")
    print("=" * 60)
    print(f"  🎤 Heard:        \"{heard}\"")
    print(f"  ✅ Matched:      {len(found_words)} → {found_words}")
    print(f"  ⏭️  Skipped:      {len(skipped)} stop words ({skipped[:5]}...)")
    print(f"  ❌ Not found:    {len(not_found)} ({not_found[:5]}...)")
    print(f"  📊 Match rate:   {len(found_words)/max(len(all_words)-len(skipped),1)*100:.0f}%")

    if not found_words:
        print(f"\n  💡 Tip: Try these available words:")
        print(f"     {sorted(available_words)[:20]}...")
    print("=" * 60)

    return found_words


# ══════════════════════════════════════════════════════════════
#  🤟 دالة إضافية: نص → إشارة (بدون صوت)
# ══════════════════════════════════════════════════════════════
def text_to_sign(text):
    """مطابقة نص مباشرة لفيديوهات إشارة (بدون تسجيل صوت)"""
    print("=" * 60)
    print("  📝 Text → Sign Language Video")
    print("=" * 60)
    print(f"  Input: \"{text}\"")

    words = text.lower().split()
    found = []

    for word in words:
        clean = word.strip(".,!?;:'\"()[]")
        if not clean or clean in STOP_WORDS:
            continue

        if clean in available_words:
            found.append(clean)
        else:
            matches = difflib.get_close_matches(
                clean, available_words, n=1, cutoff=0.7
            )
            if matches:
                found.append(matches[0])

    print(f"  Matched: {found}\n")
    for w in found:
        get_sign_video(w, max_width=280)
    print()

    return found


print(f"\n{'=' * 60}")
print(f"  ✅ Everything ready!")
print(f"{'=' * 60}")
print(f"  Words available: {len(available_words)}")
print(f"  Whisper model:   {'faster-whisper' if USE_FASTER else 'openai-whisper'}")
print(f"  Classifier:      {'loaded' if has_model else 'NOT loaded'}")
print(f"\n  Usage:")
print(f"    speech_to_sign_final('/content/audio.wav')")
print(f"    text_to_sign('hello my friend')")
print(f"{'=' * 60}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 6.0 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [35]:
# ========================================
#  تحميل فيديوهات WLASL للكلمات الـ100 فقط
# ========================================
import os, json, urllib.request

# مسار الحفظ
VIDEOS_DIR = '/content/videos'
os.makedirs(VIDEOS_DIR, exist_ok=True)

# تحميل بيانات WLASL
wlasl_json_path = '/content/drive/MyDrive/wlasl_data/WLASL_v0.3.json'
with open(wlasl_json_path, 'r') as f:
    wlasl_data = json.load(f)

# استخراج أسماء الكلمات الـ100
with open('/content/drive/MyDrive/WLASL_Project/class_indices.json', 'r') as f:
    class_indices = json.load(f)

target_words = set(class_indices.keys())
print(f"📋 الكلمات المطلوبة: {len(target_words)} كلمة")

# جمع video_id للكلمات الـ100 فقط
video_ids_to_download = []
for entry in wlasl_data:
    if entry['gloss'].lower() in target_words:
        for instance in entry['instances']:
            video_ids_to_download.append(instance['video_id'])

print(f"📹 عدد الفيديوهات المطلوبة: {len(video_ids_to_download)}")

# عرض أول 10
print(f"أول 10 video IDs: {video_ids_to_download[:10]}")

📋 الكلمات المطلوبة: 100 كلمة
📹 عدد الفيديوهات المطلوبة: 2038
أول 10 video IDs: ['69241', '65225', '68011', '68208', '68012', '70212', '70266', '07085', '07086', '07087']


In [36]:
import os
videos_dir = '/content/drive/MyDrive/wlasl_data/videos'
if os.path.exists(videos_dir):
    videos = [f for f in os.listdir(videos_dir) if f.endswith('.mp4')]
    print(f"📁 عدد الفيديوهات: {len(videos)}")
    print(f"أول 5: {videos[:5]}")
else:
    print("❌ مجلد الفيديوهات غير موجود!")

📁 عدد الفيديوهات: 21083
أول 5: ['67887.mp4', '67871.mp4', '67956.mp4', '67803.mp4', '67751.mp4']


In [44]:
# ========================================
#  بناء قاموس الفيديوهات والتحقق
# ========================================
import os, json

VIDEOS_DIR = '/content/drive/MyDrive/wlasl_data/videos'

# تحميل بيانات WLASL
wlasl_json_path = '/content/drive/MyDrive/wlasl_data/WLASL_v0.3.json'
with open(wlasl_json_path, 'r') as f:
    wlasl_data = json.load(f)

# تحميل الكلمات الـ100
with open('/content/drive/MyDrive/WLASL_Project/class_indices.json', 'r') as f:
    class_indices = json.load(f)

target_words = set(class_indices.keys())

# ── بناء word_video_cache ──
word_video_cache = {}
missing_videos = []

for entry in wlasl_data:
    word = entry['gloss'].lower()
    if word in target_words:
        # نبحث عن أول فيديو موجود فعلاً
        for instance in entry['instances']:
            vid_id = instance['video_id']
            vid_path = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
            if os.path.exists(vid_path):
                word_video_cache[word] = vid_id
                break

        # إذا لم نجد أي فيديو
        if word not in word_video_cache:
            missing_videos.append(word)

# ── التقرير ──
print("=" * 60)
print("📊 تقرير قاموس الفيديوهات")
print("=" * 60)
print(f"   الكلمات الـ100:          {len(target_words)}")
print(f"   ✅ لها فيديو موجود:      {len(word_video_cache)}")
print(f"   ❌ بدون فيديو:           {len(missing_videos)}")

if missing_videos:
    print(f"\n   الكلمات بدون فيديو:")
    for w in missing_videos:
        print(f"      ⚠️ {w}")

# عرض أمثلة
print(f"\n📋 أمثلة على الكلمات وفيديوهاتها:")
for word in list(word_video_cache.keys())[:10]:
    vid_id = word_video_cache[word]
    vid_path = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
    size_mb = os.path.getsize(vid_path) / (1024*1024)
    print(f"   ✅ {word:15s} → {vid_id}.mp4 ({size_mb:.1f} MB)")

print(f"\n🎯 جاهز! {len(word_video_cache)}/100 كلمة لها فيديو")

📊 تقرير قاموس الفيديوهات
   الكلمات الـ100:          100
   ✅ لها فيديو موجود:      100
   ❌ بدون فيديو:           0

📋 أمثلة على الكلمات وفيديوهاتها:
   ✅ book            → 69241.mp4 (0.1 MB)
   ✅ drink           → 69302.mp4 (0.1 MB)
   ✅ computer        → 12306.mp4 (0.0 MB)
   ✅ before          → 05724.mp4 (0.0 MB)
   ✅ chair           → 09847.mp4 (0.0 MB)
   ✅ go              → 24857.mp4 (0.0 MB)
   ✅ clothes         → 11305.mp4 (0.0 MB)
   ✅ who             → 63219.mp4 (0.0 MB)
   ✅ candy           → 08909.mp4 (0.0 MB)
   ✅ cousin          → 65415.mp4 (0.2 MB)

🎯 جاهز! 100/100 كلمة لها فيديو


In [46]:
# ========================================
#  التحقق من صحة ملفات الفيديو
# ========================================
import os, json, cv2

VIDEOS_DIR = '/content/drive/MyDrive/wlasl_data/videos'

with open('/content/drive/MyDrive/WLASL_Project/class_indices.json', 'r') as f:
    class_indices = json.load(f)

with open('/content/drive/MyDrive/wlasl_data/WLASL_v0.3.json', 'r') as f:
    wlasl_data = json.load(f)

target_words = set(class_indices.keys())

# إعادة بناء word_video_cache
word_video_cache = {}
for entry in wlasl_data:
    word = entry['gloss'].lower()
    if word in target_words and word not in word_video_cache:
        for instance in entry['instances']:
            vid_id = instance['video_id']
            vid_path = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
            if os.path.exists(vid_path):
                word_video_cache[word] = vid_id
                break

# فحص صحة الفيديوهات
valid = 0
corrupted = []
too_small = []

for word, vid_id in word_video_cache.items():
    vid_path = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
    size_kb = os.path.getsize(vid_path) / 1024

    if size_kb < 1:  # أقل من 1 KB = ملف فارغ
        too_small.append((word, vid_id, size_kb))
        continue

    # فحص بـ OpenCV
    cap = cv2.VideoCapture(vid_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    if frame_count > 0:
        valid += 1
    else:
        corrupted.append((word, vid_id, size_kb))

print("=" * 60)
print("📊 تقرير صحة الفيديوهات")
print("=" * 60)
print(f"   ✅ صالحة:           {valid}")
print(f"   ⚠️ صغيرة جداً:      {len(too_small)}")
print(f"   ❌ تالفة:           {len(corrupted)}")

if too_small:
    print(f"\n📋 ملفات صغيرة جداً (قد تكون فارغة):")
    for word, vid_id, size in too_small[:10]:
        print(f"   ⚠️ {word:15s} → {vid_id}.mp4 ({size:.1f} KB)")

if corrupted:
    print(f"\n📋 ملفات تالفة:")
    for word, vid_id, size in corrupted[:10]:
        print(f"   ❌ {word:15s} → {vid_id}.mp4 ({size:.1f} KB)")

📊 تقرير صحة الفيديوهات
   ✅ صالحة:           100
   ⚠️ صغيرة جداً:      0
   ❌ تالفة:           0


In [47]:
# ========================================
#  🚀 Pipeline كامل: صوت → نص → إشارة
# ========================================
import os, json, difflib, cv2
from base64 import b64encode, b64decode
from IPython.display import HTML, Javascript, display
from google.colab import output
import io

# ── المسارات ──
VIDEOS_DIR = '/content/drive/MyDrive/wlasl_data/videos'
WLASL_JSON = '/content/drive/MyDrive/wlasl_data/WLASL_v0.3.json'
CLASS_JSON = '/content/drive/MyDrive/WLASL_Project/class_indices.json'
SIGN_VIDEO_DIR = '/content/drive/MyDrive/WLASL_Project/sign_videos'
os.makedirs(SIGN_VIDEO_DIR, exist_ok=True)

# ── تحميل البيانات ──
with open(CLASS_JSON, 'r') as f:
    class_indices = json.load(f)
with open(WLASL_JSON, 'r') as f:
    wlasl_data = json.load(f)

available_words = list(class_indices.keys())
target_words = set(available_words)

# ── بناء word_video_cache ──
word_video_cache = {}
for entry in wlasl_data:
    word = entry['gloss'].lower()
    if word in target_words and word not in word_video_cache:
        for instance in entry['instances']:
            vid_id = instance['video_id']
            vid_path = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
            if os.path.exists(vid_path):
                # تحقق أن الملف صالح
                cap = cv2.VideoCapture(vid_path)
                frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                cap.release()
                if frames > 0:
                    word_video_cache[word] = vid_id
                    break

# ── كلمات الربط ──
STOP_WORDS = {
    'a', 'an', 'the', 'is', 'am', 'are', 'was', 'were', 'be',
    'to', 'of', 'in', 'for', 'on', 'at', 'by', 'with', 'from',
    'and', 'or', 'if', 'so', 'not', 'do',
    'my', 'your', 'his', 'her', 'its', 'our', 'their',
    'this', 'that', 'it', 'i', 'me', 'you', 'he', 'she',
    'we', 'they', 'have', 'has', 'had', 'will', 'would',
    'could', 'should', 'may', 'might', 'shall'
}

# ══════════════════════════════════════════════════════════════
#  🤟 دالة عرض فيديو الإشارة
# ══════════════════════════════════════════════════════════════
def show_sign_video(word, max_width=320):
    vid_id = word_video_cache.get(word.lower())
    if not vid_id:
        print(f"  ❌ '{word}' not in cache")
        return None

    src_path = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
    if not os.path.exists(src_path):
        print(f"  ❌ Video '{vid_id}.mp4' not found")
        return None

    web_path = os.path.join(SIGN_VIDEO_DIR, f'{word.lower()}.mp4')
    if not os.path.exists(web_path):
        os.system(
            f'ffmpeg -y -i "{src_path}" '
            f'-c:v libx264 -preset fast -crf 23 -an '
            f'-vf "scale={max_width}:-2" "{web_path}" '
            f'> /dev/null 2>&1'
        )

    try:
        with open(web_path, 'rb') as f:
            mp4 = b64encode(f.read()).decode()
        display(HTML(
            f'<div style="display:inline-block; margin:10px; '
            f'text-align:center; background:#f5f5f5; '
            f'padding:15px; border-radius:12px; '
            f'box-shadow: 0 2px 8px rgba(0,0,0,0.1);">'
            f'<b style="color:#1a73e8; font-size:18px;">🤟 {word.upper()}</b><br>'
            f'<video width="{max_width}" controls autoplay>'
            f'<source src="data:video/mp4;base64,{mp4}" type="video/mp4">'
            f'</video></div>'
        ))
        return web_path
    except Exception as e:
        print(f"  ❌ Display error: {e}")
        return None

# ══════════════════════════════════════════════════════════════
#  🎙️ دالة التسجيل من الميكروفون
# ══════════════════════════════════════════════════════════════
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onload = () => resolve(reader.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({audio: true})
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async () => {
    track = stream.getTracks()[0]
    track.stop()
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record_audio(seconds=7):
    print(f"\n🎙️ سجّل الآن... ({seconds} ثانية)")
    display(Javascript(RECORD_JS))
    s = output.eval_js(f'record({seconds * 1000})')
    b = b64decode(s.split(',')[1])
    from pydub import AudioSegment
    audio = AudioSegment.from_file(io.BytesIO(b))
    audio_path = '/content/recorded_audio.wav'
    audio.export(audio_path, format='wav')
    print(f"✅ تم التسجيل! ({len(audio)/1000:.1f} ثانية)")
    return audio_path

# ══════════════════════════════════════════════════════════════
#  🔍 دالة مطابقة الكلمات
# ══════════════════════════════════════════════════════════════
def match_words(heard, available_words, stop_words, threshold=0.7):
    all_words = heard.split()
    found_words = []
    skipped = []
    not_found = []

    for word in all_words:
        clean = word.strip(".,!?;:'\"()[]").lower()
        if not clean or clean in stop_words or len(clean) <= 1:
            skipped.append(clean)
            continue

        if clean in available_words:
            found_words.append(clean)
            print(f"   ✅ '{clean}' → إشارة معروفة")
        else:
            matches = difflib.get_close_matches(clean, available_words, n=1, cutoff=threshold)
            if matches:
                sim = difflib.SequenceMatcher(None, clean, matches[0]).ratio()
                found_words.append(matches[0])
                print(f"   🔀 '{clean}' → '{matches[0]}' ({sim:.0%} similar)")
            else:
                not_found.append(clean)
                print(f"   🔤 '{clean}' → غير معروفة")

    return found_words, skipped, not_found

# ── تحميل Whisper ──
print("⏳ تحميل Whisper...")
try:
    from faster_whisper import WhisperModel
    wm = WhisperModel("base", device="auto", compute_type="int8")
    USE_FASTER = True
except:
    import whisper
    wm = whisper.load_model("base")
    USE_FASTER = False

print(f"✅ Whisper: {'faster-whisper' if USE_FASTER else 'openai-whisper'}")
print(f"✅ فيديوهات: {len(word_video_cache)}/100 كلمة")
print(f"✅ Pipeline جاهز!")

⏳ تحميل Whisper...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 100MiB/s]


✅ Whisper: openai-whisper
✅ فيديوهات: 100/100 كلمة
✅ Pipeline جاهز!


In [61]:
# ================================================================
# 🎙️ Record a Sentence (click button) - الإصدار المعدل
# ================================================================
import os, base64
from google.colab import output
from IPython.display import HTML, display

# تنظيف تسجيل قديم
for old_file in ['/content/recorded_speech.webm', '/content/recorded_speech.wav']:
    if os.path.exists(old_file):
        os.remove(old_file)

def saveAudio(data):
    audio = base64.b64decode(data)
    # حفظ بصيغة WebM الصحيحة
    filepath = '/content/recorded_speech.webm'
    with open(filepath, 'wb') as f:
        f.write(audio)

    size = os.path.getsize(filepath)
    if size < 1000:
        print("⚠️ Recording seems empty! Try again.")
        return

    # تحويل WebM → WAV باستخدام ffmpeg
    wav_path = '/content/recorded_speech.wav'
    ret = os.system(
        f'ffmpeg -y -i "{filepath}" -ar 16000 -ac 1 '
        f'"{wav_path}" > /dev/null 2>&1'
    )

    if ret == 0 and os.path.exists(wav_path):
        wav_size = os.path.getsize(wav_path) / 1024
        print(f"✅ Recording saved! ({wav_size:.1f} KB)")
        print("   → Run the pipeline cell to process.")
    else:
        print("⚠️ Conversion failed. Using webm directly.")
        print(f"✅ Recording saved as webm ({size/1024:.1f} KB)")

output.register_callback('saveAudio', saveAudio)

display(HTML("""
<div style="padding: 20px; border: 2px solid #e0e0e0; border-radius: 12px;
            text-align: center; max-width: 500px; margin: 10px auto;
            box-shadow: 0 2px 8px rgba(0,0,0,0.1);">
    <h2 style="color: #333; margin-bottom: 5px;">🎙️ Record a Sentence</h2>
    <p style="color: #777; margin-bottom: 15px;">Click record, then speak clearly</p>

    <div style="display: flex; gap: 10px; justify-content: center; margin-bottom: 10px;">
        <button id="recBtn" onclick="toggleRecording()"
            style="padding: 14px 32px; font-size: 16px; background: #4CAF50;
                   color: white; border: none; border-radius: 25px;
                   cursor: pointer; font-weight: bold; min-width: 180px;
                   transition: all 0.3s;">
            Start Recording
        </button>
        <button id="stopBtn" onclick="stopRecording()" disabled
            style="padding: 14px 24px; font-size: 16px; background: #ccc;
                   color: #666; border: none; border-radius: 25px;
                   cursor: not-allowed; font-weight: bold; min-width: 120px;">
            Stop
        </button>
    </div>

    <p id="status" style="margin: 10px 0; font-weight: bold; color: #666;"></p>
    <p id="timer" style="font-size: 24px; font-weight: bold; color: #f44336; display: none;">
        0:00
    </p>
</div>

<script>
let recording = false;
let mediaRecorder = null;
let timerInterval = null;
let seconds = 0;

function updateTimer() {
    seconds++;
    const m = Math.floor(seconds / 60);
    const s = seconds % 60;
    document.getElementById('timer').textContent =
        m + ':' + (s < 10 ? '0' : '') + s;
}

async function toggleRecording() {
    if (recording) return;
    recording = true;

    const btn = document.getElementById('recBtn');
    const stopBtn = document.getElementById('stopBtn');
    const status = document.getElementById('status');
    const timer = document.getElementById('timer');

    btn.disabled = true;
    btn.style.background = '#f44336';
    btn.textContent = 'Recording...';
    stopBtn.disabled = false;
    stopBtn.style.background = '#FF9800';
    stopBtn.style.color = 'white';
    stopBtn.style.cursor = 'pointer';
    status.textContent = '🔴 Speak now!';
    status.style.color = '#f44336';
    timer.style.display = 'block';
    timer.textContent = '0:00';
    seconds = 0;
    timerInterval = setInterval(updateTimer, 1000);

    try {
        const stream = await navigator.mediaDevices.getUserMedia({
            audio: {
                echoCancellation: true,
                noiseSuppression: true,
                sampleRate: 16000
            }
        });

        mediaRecorder = new MediaRecorder(stream, {
            mimeType: 'audio/webm;codecs=opus'
        });

        const chunks = [];
        mediaRecorder.ondataavailable = e => chunks.push(e.data);

        mediaRecorder.onstop = () => {
            clearInterval(timerInterval);
            stream.getTracks().forEach(t => t.stop());

            const blob = new Blob(chunks, {type: 'audio/webm'});

            // ── طريقة آمنة للتحويل إلى base64 ──
            const reader = new FileReader();
            reader.onload = () => {
                const arrayBuffer = reader.result;
                const bytes = new Uint8Array(arrayBuffer);

                // تحويل آمن على دفعات (لا يسبب Stack Overflow)
                let binary = '';
                const chunkSize = 8192;
                for (let i = 0; i < bytes.length; i += chunkSize) {
                    const chunk = bytes.subarray(i, i + chunkSize);
                    binary += String.fromCharCode.apply(null, chunk);
                }
                const base64 = btoa(binary);

                google.colab.kernel.invokeFunction('saveAudio', [base64], {});
            };
            reader.readAsArrayBuffer(blob);

            // إعادة تعيين الأزرار
            btn.disabled = false;
            btn.style.background = '#4CAF50';
            btn.textContent = 'Start Recording';
            stopBtn.disabled = true;
            stopBtn.style.background = '#ccc';
            stopBtn.style.color = '#666';
            stopBtn.style.cursor = 'not-allowed';
            status.textContent = '✅ Saved! Run the pipeline cell.';
            status.style.color = '#4CAF50';
            recording = false;
        };

        // إيقاف تلقائي بعد 10 ثوانٍ
        setTimeout(() => {
            if (recording) stopRecording();
        }, 10000);

        mediaRecorder.start();
    } catch(err) {
        const status = document.getElementById('status');
        status.textContent = 'Mic error: ' + err.message;
        status.style.color = '#f44336';
        recording = false;
    }
}

function stopRecording() {
    if (mediaRecorder && mediaRecorder.state !== 'inactive') {
        mediaRecorder.stop();
    }
}
</script>
"""))

print("💡 Try saying one of these sentences:")
print("  " + "-" * 40)
examples = [
    "I like hot drink",
    "who can help",
    "walk dog now",
    "want eat pizza",
    "yes or no",
    "book and computer",
]
for ex in examples:
    print(f"  🗣️  \"{ex}\"")
print("  " + "-" * 40)

💡 Try saying one of these sentences:
  ----------------------------------------
  🗣️  "I like hot drink"
  🗣️  "who can help"
  🗣️  "walk dog now"
  🗣️  "want eat pizza"
  🗣️  "yes or no"
  🗣️  "book and computer"
  ----------------------------------------
✅ Recording saved! (120.1 KB)
   → Run the pipeline cell to process.


In [63]:
# ================================================================
#  🚀 Pipeline: صوت → نص → معالجة → فيديو إشارة
# ================================================================
import os, json, difflib, cv2
from base64 import b64encode
from IPython.display import HTML, display

# ── المسارات ──
VIDEOS_DIR = '/content/drive/MyDrive/wlasl_data/videos'
WLASL_JSON = '/content/drive/MyDrive/wlasl_data/WLASL_v0.3.json'
CLASS_JSON = '/content/drive/MyDrive/WLASL_Project/class_indices.json'
SIGN_DIR   = '/content/drive/MyDrive/WLASL_Project/sign_videos'
os.makedirs(SIGN_DIR, exist_ok=True)

# ── تحميل البيانات ──
with open(CLASS_JSON, 'r') as f:
    class_indices = json.load(f)
with open(WLASL_JSON, 'r') as f:
    wlasl_data = json.load(f)

available_words = list(class_indices.keys())
target_words = set(available_words)

# ── بناء word_video_cache ──
word_video_cache = {}
for entry in wlasl_data:
    word = entry['gloss'].lower()
    if word in target_words and word not in word_video_cache:
        for instance in entry['instances']:
            vid_id = instance['video_id']
            vid_path = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
            if os.path.exists(vid_path):
                cap = cv2.VideoCapture(vid_path)
                frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                cap.release()
                if frames > 0:
                    word_video_cache[word] = vid_id
                    break

# ── كلمات الربط ──
STOP_WORDS = {
    'a', 'an', 'the', 'is', 'am', 'are', 'was', 'were', 'be',
    'to', 'of', 'in', 'for', 'on', 'at', 'by', 'with', 'from',
    'and', 'or', 'if', 'so', 'not', 'do',
    'my', 'your', 'his', 'her', 'its', 'our', 'their',
    'this', 'that', 'it', 'i', 'me', 'you', 'he', 'she',
    'we', 'they', 'have', 'has', 'had', 'will', 'would',
    'could', 'should', 'may', 'might', 'shall'
}

# ══════════════════════════════════════════════════════════════
#  الدوال المساعدة
# ══════════════════════════════════════════════════════════════

def show_sign_video(word, max_width=280):
    """عرض فيديو إشارة لكلمة واحدة"""
    vid_id = word_video_cache.get(word.lower())
    if not vid_id:
        return None

    src_path = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
    web_path = os.path.join(SIGN_DIR, f'{word.lower()}.mp4')

    if not os.path.exists(web_path):
        os.system(
            f'ffmpeg -y -i "{src_path}" '
            f'-c:v libx264 -preset fast -crf 23 -an '
            f'-vf "scale={max_width}:-2" "{web_path}" '
            f'> /dev/null 2>&1'
        )

    try:
        with open(web_path, 'rb') as f:
            mp4 = b64encode(f.read()).decode()
        display(HTML(
            f'<div style="display:inline-block; margin:8px; text-align:center; '
            f'background:linear-gradient(135deg,#667eea,#764ba2); '
            f'padding:12px 15px 8px; border-radius:14px; color:white;">'
            f'<div style="font-size:16px; font-weight:bold; margin-bottom:5px;">'
            f'🤟 {word.upper()}</div>'
            f'<video width="{max_width}" controls autoplay '
            f'style="border-radius:8px;">'
            f'<source src="data:video/mp4;base64,{mp4}" type="video/mp4">'
            f'</video></div>'
        ))
        return web_path
    except:
        return None


def match_words(heard):
    """مطابقة الكلمات مع القاموس"""
    found, skipped, not_found = [], [], []

    for word in heard.split():
        clean = word.strip(".,!?;:'\"()[]").lower()
        if not clean or clean in STOP_WORDS or len(clean) <= 1:
            skipped.append(clean)
            continue

        if clean in available_words:
            found.append(clean)
        else:
            matches = difflib.get_close_matches(clean, available_words, n=1, cutoff=0.7)
            if matches:
                found.append(matches[0])
            else:
                not_found.append(clean)

    return found, skipped, not_found


def transcribe_audio(audio_path):
    """تحويل الصوت إلى نص"""
    try:
        from faster_whisper import WhisperModel
        wm = WhisperModel("base", device="auto", compute_type="int8")
        segments, _ = wm.transcribe(audio_path, language="en",
                                     beam_size=5, vad_filter=True)
        return " ".join(seg.text for seg in segments).strip().lower()
    except:
        import whisper
        wm = whisper.load_model("base")
        result = wm.transcribe(audio_path, language="en")
        return result["text"].strip().lower()


# ══════════════════════════════════════════════════════════════
#  🚀 الـ Pipeline الرئيسي
# ══════════════════════════════════════════════════════════════

def pipeline(audio_path):
    """
    Pipeline كامل: صوت → نص → معالجة → فيديو إشارة
    """
    display(HTML(
        '<div style="text-align:center; padding:10px; margin:10px 0; '
        'background:linear-gradient(90deg,#1a73e8,#34a853); '
        'border-radius:10px; color:white; font-size:20px; font-weight:bold;">'
        '🚀 Speech → Sign Language Pipeline</div>'
    ))

    # ── المرحلة 1: تحويل الصوت إلى نص ──
    print("🎤 [1/3] Transcribing audio...")
    heard = transcribe_audio(audio_path)

    if not heard:
        print("❌ No speech detected! Try recording again.")
        return

    display(HTML(
        f'<div style="padding:12px; margin:10px 0; background:#e8f5e9; '
        f'border-radius:10px; border-left:4px solid #4CAF50;">'
        f'<b>🎤 Heard:</b> <span style="font-size:18px;">"{heard}"</span></div>'
    ))

    # ── المرحلة 2: مطابقة الكلمات ──
    print("🔍 [2/3] Matching words...")
    found, skipped, not_found = match_words(heard)

    # بناء HTML للنتائج
    results_html = '<div style="padding:12px; margin:10px 0; background:#fff3e0; border-radius:10px; border-left:4px solid #FF9800;">'
    results_html += '<b>📊 Word Analysis:</b><br>'

    for word in heard.split():
        clean = word.strip(".,!?;:'\"()[]").lower()
        if clean in STOP_WORDS or len(clean) <= 1:
            results_html += f' <span style="background:#e0e0e0; padding:3px 8px; border-radius:12px; margin:2px; font-size:14px;">⏭️ {clean}</span>'
        elif clean in available_words:
            results_html += f' <span style="background:#c8e6c9; padding:3px 8px; border-radius:12px; margin:2px; font-size:14px;">✅ {clean}</span>'
        else:
            matches = difflib.get_close_matches(clean, available_words, n=1, cutoff=0.7)
            if matches:
                results_html += f' <span style="background:#fff9c4; padding:3px 8px; border-radius:12px; margin:2px; font-size:14px;">🔀 {clean}→{matches[0]}</span>'
            else:
                results_html += f' <span style="background:#ffcdd2; padding:3px 8px; border-radius:12px; margin:2px; font-size:14px;">🔤 {clean}</span>'

    results_html += '</div>'
    display(HTML(results_html))

    # ── المرحلة 3: عرض فيديوهات الإشارة ──
    print(f"🎬 [3/3] Playing {len(found)} sign video(s)...")

    if found:
        display(HTML(
            '<div style="text-align:center; margin:10px 0;">'
            '<h3 style="color:#1a73e8;">🤟 Sign Language Translation</h3>'
            '<div style="display:flex; flex-wrap:wrap; justify-content:center;">'
        ))
        for word in found:
            show_sign_video(word)
        display(HTML('</div></div>'))
    else:
        display(HTML(
            '<div style="padding:15px; margin:10px; background:#ffebee; '
            'border-radius:10px; text-align:center;">'
            '<b>❌ No matching signs found!</b><br>'
            'Try using words from the vocabulary list.</div>'
        ))

    # ── التقرير النهائي ──
    content = len(heard.split()) - len(skipped)
    rate = f"{len(found)/content*100:.0f}%" if content > 0 else "0%"

    display(HTML(
        f'<div style="padding:15px; margin:10px 0; background:#e3f2fd; '
        f'border-radius:10px; border-left:4px solid #1a73e8;">'
        f'<b>📋 Final Report</b><br>'
        f'🎤 Heard: <b>"{heard}"</b><br>'
        f'✅ Signs found: <b>{found}</b> ({len(found)})<br>'
        f'⏭️ Skipped: {skipped[:5]}<br>'
        f'🔤 Unknown: {not_found[:5]}<br>'
        f'📊 Translation rate: <b style="font-size:18px; color:#1a73e8;">{rate}</b>'
        f'</div>'
    ))


# ══════════════════════════════════════════════════════════════
#  تشغيل الـ Pipeline
# ══════════════════════════════════════════════════════════════

# البحث عن ملف الصوت
audio_file = None
for path in ['/content/recorded_speech.wav',
             '/content/recorded_speech.webm',
             '/content/recorded_audio.wav']:
    if os.path.exists(path):
        audio_file = path
        size = os.path.getsize(path) / 1024
        print(f"📁 Found audio: {path} ({size:.1f} KB)")
        break

if audio_file:
    pipeline(audio_file)
else:
    print("❌ No audio file found!")
    print("   → Run the recording cell first, then come back here.")

📁 Found audio: /content/recorded_speech.wav (120.1 KB)


🎤 [1/3] Transcribing audio...


🔍 [2/3] Matching words...


🎬 [3/3] Playing 3 sign video(s)...


In [65]:
# ================================================================
#  🎬 دمج فيديوهات الإشارة في فيديو واحد
# ================================================================
import os, json, difflib, cv2
from base64 import b64encode
from IPython.display import HTML, display
from moviepy.editor import VideoFileClip, ImageClip, concatenate_videoclips
import numpy as np

VIDEOS_DIR = '/content/drive/MyDrive/wlasl_data/videos'
SIGN_DIR = '/content/drive/MyDrive/WLASL_Project/sign_videos'
FINAL_DIR = '/content/drive/MyDrive/WLASL_Project/final_videos'
os.makedirs(FINAL_DIR, exist_ok=True)

# ── دالة إنشاء بطاقة كلمة (انتقالية) ──
def create_word_card(word, duration=1.5, size=(640, 360)):
    """إنشاء صورة عنوان للكلمة قبل عرض الإشارة"""
    img = np.zeros((size[1], size[0], 3), dtype=np.uint8)

    # خلفية متدرجة
    for y in range(size[1]):
        r = int(26 + (52 - 26) * y / size[1])
        g = int(115 + (168 - 115) * y / size[1])
        b = int(232 + (83 - 232) * y / size[1])
        img[y, :] = [r, g, b]

    # كتابة الكلمة
    font = cv2.FONT_HERSHEY_SIMPLEX
    text_size = cv2.getTextSize(word.upper(), font, 3.0, 5)[0]
    text_x = (size[0] - text_size[0]) // 2
    text_y = (size[1] + text_size[1]) // 2

    cv2.putText(img, word.upper(), (text_x, text_y), font, 3.0, (255, 255, 255), 5)

    # أيقونة إشارة
    cv2.putText(img, "Sign:", (20, 50), font, 1.0, (200, 230, 255), 2)

    clip = ImageClip(img).set_duration(duration)
    return clip

# ── دالة إنشاء تهجئة أبجدية ──
def create_fingerspell_clip(word, duration_per_letter=0.8, size=(640, 360)):
    """إنشاء مقطع تهجئة لكلمة غير معروفة"""
    clips = []

    for letter in word.upper():
        img = np.zeros((size[1], size[0], 3), dtype=np.uint8)

        # خلفية برتقالية
        for y in range(size[1]):
            img[y, :] = [30, 80 + int(50 * y / size[1]), 200 + int(55 * y / size[1])]

        # الحرف
        font = cv2.FONT_HERSHEY_SIMPLEX
        text_size = cv2.getTextSize(letter, font, 6.0, 10)[0]
        text_x = (size[0] - text_size[0]) // 2
        text_y = (size[1] + text_size[1]) // 2

        cv2.putText(img, letter, (text_x, text_y), font, 6.0, (255, 255, 255), 10)
        cv2.putText(img, "Fingerspelling", (20, 40), font, 0.8, (200, 200, 200), 2)
        cv2.putText(img, word.upper(), (20, size[1] - 20), font, 0.8, (180, 180, 180), 1)

        clip = ImageClip(img).set_duration(duration_per_letter)
        clips.append(clip)

    return clips

# ── دالة إنشاء شارة البداية ──
def create_intro_clip(duration=2.0, size=(640, 360)):
    """شارة بداية الفيديو"""
    img = np.zeros((size[1], size[0], 3), dtype=np.uint8)

    for y in range(size[1]):
        r = int(26 + (100 - 26) * y / size[1])
        g = int(115 + (50 - 115) * y / size[1])
        b = int(232 + (150 - 232) * y / size[1])
        img[y, :] = [r, g, b]

    font = cv2.FONT_HERSHEY_SIMPLEX

    texts = [
        ("Speech to Sign Language", (size[0]//2 - 230, size[1]//2 - 30), 1.2, (255, 255, 255), 3),
        ("Translation", (size[0]//2 - 100, size[1]//2 + 30), 1.0, (200, 230, 255), 2),
    ]

    for text, pos, scale, color, thick in texts:
        cv2.putText(img, text, pos, font, scale, color, thick)

    return ImageClip(img).set_duration(duration)

# ── دالة إنشاء شارة النهاية ──
def create_outro_clip(duration=2.0, size=(640, 360)):
    """شارة نهاية الفيديو"""
    img = np.zeros((size[1], size[0], 3), dtype=np.uint8)

    for y in range(size[1]):
        img[y, :] = [int(30 + 50 * y / size[1]), int(100 + 50 * y / size[1]), 30]

    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(img, "End of Translation", (size[0]//2 - 180, size[1]//2),
                font, 1.2, (255, 255, 255), 3)

    return ImageClip(img).set_duration(duration)


# ══════════════════════════════════════════════════════════════
#  🎬 الدالة الرئيسية: دمج فيديوهات الإشارة
# ══════════════════════════════════════════════════════════════

def create_sign_video(found_words, not_found_words=None, heard_text="",
                       output_name="sign_translation.mp4", size=(640, 360)):
    """
    دمج فيديوهات الإشارة في فيديو واحد متصل

    المعاملات:
        found_words: الكلمات المعروفة (لها فيديو إشارة)
        not_found_words: الكلمات غير المعروفة (ستُهجأ)
        heard_text: النص الأصلي (للعرض في البداية)
        output_name: اسم ملف الخرج
        size: حجم الفيديو
    """
    clips = []

    # شارة البداية
    intro = create_intro_clip(duration=2.0, size=size)
    clips.append(intro)

    # ── إضافة فيديوهات الكلمات المعروفة ──
    for word in found_words:
        # بطاقة اسم الكلمة
        card = create_word_card(word, duration=1.0, size=size)
        clips.append(card)

        # فيديو الإشارة
        vid_id = word_video_cache.get(word.lower())
        if vid_id:
            src_path = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
            processed_path = os.path.join(SIGN_DIR, f'{word.lower()}.mp4')

            # تحويل إذا لم يكن موجوداً
            if not os.path.exists(processed_path):
                os.system(
                    f'ffmpeg -y -i "{src_path}" '
                    f'-c:v libx264 -preset fast -crf 23 -an '
                    f'-vf "scale={size[0]}:{size[1]}" "{processed_path}" '
                    f'> /dev/null 2>&1'
                )

            if os.path.exists(processed_path):
                try:
                    sign_clip = VideoFileClip(processed_path)
                    # تقليص المدة إذا طالت
                    if sign_clip.duration > 4.0:
                        sign_clip = sign_clip.subclip(0, 4.0)
                    clips.append(sign_clip)
                except:
                    pass

    # ── تهجئة الكلمات غير المعروفة ──
    if not_found_words:
        for word in not_found_words:
            spell_clips = create_fingerspell_clip(word, size=size)
            clips.extend(spell_clips)

    # شارة النهاية
    outro = create_outro_clip(duration=1.5, size=size)
    clips.append(outro)

    # ── دمج كل المقاطع ──
    if clips:
        final = concatenate_videoclips(clips, method='compose')
        output_path = os.path.join(FINAL_DIR, output_name)
        final.write_videofile(output_path, fps=24, codec='libx264',
                              audio=False, logger=None)

        # عرض الفيديو النهائي
        with open(output_path, 'rb') as f:
            mp4 = b64encode(f.read()).decode()

        display(HTML(
            f'<div style="text-align:center; margin:20px; padding:15px; '
            f'background:#f0f0f0; border-radius:15px;">'
            f'<h3 style="color:#1a73e8;">🎬 Final Sign Language Video</h3>'
            f'<p>Duration: {final.duration:.1f}s | Words: {len(found_words)} signs + {len(not_found_words)} spelled</p>'
            f'<video width="640" controls autoplay style="border-radius:10px;">'
            f'<source src="data:video/mp4;base64,{mp4}" type="video/mp4">'
            f'</video><br>'
            f'<small>📁 {output_path}</small></div>'
        ))

        return output_path
    else:
        print("❌ No clips to merge!")
        return None


print("✅ Video merging functions ready!")
print("💡 Usage: create_sign_video(found_words, not_found_words)")

✅ Video merging functions ready!
💡 Usage: create_sign_video(found_words, not_found_words)


In [67]:
# ================================================================
#  🚀 Pipeline V2: صوت → نص → فيديو إشارة متتالي
# ================================================================
import os, json, difflib, cv2, numpy as np
from base64 import b64encode
from IPython.display import HTML, display
from moviepy.editor import VideoFileClip, ImageClip, concatenate_videoclips

# ── المسارات ──
VIDEOS_DIR = '/content/drive/MyDrive/wlasl_data/videos'
WLASL_JSON = '/content/drive/MyDrive/wlasl_data/WLASL_v0.3.json'
CLASS_JSON = '/content/drive/MyDrive/WLASL_Project/class_indices.json'
SIGN_DIR = '/content/drive/MyDrive/WLASL_Project/sign_videos'
FINAL_DIR = '/content/drive/MyDrive/WLASL_Project/final_videos'
os.makedirs(SIGN_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

# ── تحميل البيانات ──
with open(CLASS_JSON, 'r') as f:
    class_indices = json.load(f)
with open(WLASL_JSON, 'r') as f:
    wlasl_data = json.load(f)

available_words = list(class_indices.keys())
target_words = set(available_words)

# ── بناء word_video_cache ──
word_video_cache = {}
for entry in wlasl_data:
    word = entry['gloss'].lower()
    if word in target_words and word not in word_video_cache:
        for instance in entry['instances']:
            vid_id = instance['video_id']
            vid_path = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
            if os.path.exists(vid_path):
                cap = cv2.VideoCapture(vid_path)
                frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                cap.release()
                if frames > 0:
                    word_video_cache[word] = vid_id
                    break

# ── كلمات الربط ──
STOP_WORDS = {
    'a', 'an', 'the', 'is', 'am', 'are', 'was', 'were', 'be',
    'to', 'of', 'in', 'for', 'on', 'at', 'by', 'with', 'from',
    'and', 'or', 'if', 'so', 'not', 'do',
    'my', 'your', 'his', 'her', 'its', 'our', 'their',
    'this', 'that', 'it', 'i', 'me', 'you', 'he', 'she',
    'we', 'they', 'have', 'has', 'had', 'will', 'would',
    'could', 'should', 'may', 'might', 'shall'
}

# ── تحويل الأزمنة ──
lemma_map = {
    'walked': 'walk', 'walking': 'walk', 'walks': 'walk',
    'ate': 'eat', 'eating': 'eat', 'eats': 'eat',
    'drank': 'drink', 'drinking': 'drink', 'drinks': 'drink',
    'played': 'play', 'playing': 'play', 'plays': 'play',
    'liked': 'like', 'liking': 'like', 'likes': 'like',
    'worked': 'work', 'working': 'work', 'works': 'work',
    'studied': 'study', 'studying': 'study', 'studies': 'study',
    'danced': 'dance', 'dancing': 'dance', 'dances': 'dance',
    'cooked': 'cook', 'cooking': 'cook', 'cooks': 'cook',
    'gave': 'give', 'giving': 'give', 'gives': 'give',
    'told': 'tell', 'telling': 'tell', 'tells': 'tell',
    'met': 'meet', 'meeting': 'meet', 'meets': 'meet',
    'needed': 'need', 'needing': 'need', 'needs': 'need',
    'enjoyed': 'enjoy', 'enjoying': 'enjoy', 'enjoys': 'enjoy',
    'forgot': 'forget', 'forgetting': 'forget', 'forgets': 'forget',
    'helped': 'help', 'helping': 'help', 'helps': 'help',
    'wanted': 'want', 'wanting': 'want', 'wants': 'want',
    'painted': 'paint', 'painting': 'paint', 'paints': 'paint',
    'changed': 'change', 'changing': 'change', 'changes': 'change',
    'decided': 'decide', 'deciding': 'decide', 'decides': 'decide',
    'finished': 'finish', 'finishing': 'finish', 'finishes': 'finish',
    'graduated': 'graduate', 'graduating': 'graduate',
    'pulled': 'pull', 'pulling': 'pull', 'pulls': 'pull',
}

# ── تحميل Whisper ──
try:
    from faster_whisper import WhisperModel
    wm = WhisperModel("base", device="auto", compute_type="int8")
    USE_FASTER = True
except:
    import whisper
    wm = whisper.load_model("base")
    USE_FASTER = False

print(f"✅ Whisper: {'faster-whisper' if USE_FASTER else 'openai-whisper'}")
print(f"✅ Video cache: {len(word_video_cache)}/100 words")

# ══════════════════════════════════════════════════════════════
#  الدوال
# ══════════════════════════════════════════════════════════════

def transcribe_audio(audio_path):
    if not audio_path:
        return ""
    try:
        if USE_FASTER:
            segments, _ = wm.transcribe(audio_path, language="en",
                                         beam_size=5, vad_filter=True)
            return " ".join(seg.text for seg in segments).strip().lower()
        else:
            result = wm.transcribe(audio_path, language="en")
            return result["text"].strip().lower()
    except:
        return ""


def match_words(heard):
    found, skipped, not_found = [], [], []
    for word in heard.split():
        clean = word.strip(".,!?;:'\"()[]").lower()
        if clean in lemma_map:
            clean = lemma_map[clean]
        if not clean or clean in STOP_WORDS or len(clean) <= 1:
            skipped.append(clean)
            continue
        if clean in available_words:
            found.append(clean)
        else:
            matches = difflib.get_close_matches(clean, available_words, n=1, cutoff=0.7)
            if matches:
                found.append(matches[0])
            else:
                not_found.append(clean)
    return found, skipped, not_found


def create_sign_video(found, not_found, size=(640, 360)):
    """
    فيديو إشارة متتالي - بدون كتابة الكلمات
    كل فيديو يتبعه الآخر مباشرة مع انتقال أسود قصير
    """
    clips = []

    # ── كلمات معروفة: فيديو إشارة فقط ──
    for word in found:
        vid_id = word_video_cache.get(word.lower())
        if vid_id:
            processed = os.path.join(SIGN_DIR, f'{word.lower()}.mp4')
            if not os.path.exists(processed):
                src = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
                os.system(
                    f'ffmpeg -y -i "{src}" -c:v libx264 -preset fast '
                    f'-crf 23 -an -vf "scale={size[0]}:{size[1]}" '
                    f'"{processed}" > /dev/null 2>&1'
                )
            if os.path.exists(processed):
                try:
                    clip = VideoFileClip(processed)
                    if clip.duration > 4.0:
                        clip = clip.subclip(0, 4.0)
                    clips.append(clip)
                except:
                    pass

        # انتقال أسود قصير (0.3 ثانية) بين الإشارات
        black = np.zeros((size[1], size[0], 3), dtype=np.uint8)
        clips.append(ImageClip(black).set_duration(0.3))

    # ── كلمات غير معروفة: تهجئة ──
    for word in not_found:
        for letter in word.upper():
            img = np.zeros((size[1], size[0], 3), dtype=np.uint8)
            font = cv2.FONT_HERSHEY_SIMPLEX
            ts = cv2.getTextSize(letter, font, 6.0, 10)[0]
            cv2.putText(img, letter,
                        ((size[0] - ts[0]) // 2, (size[1] + ts[1]) // 2),
                        font, 6.0, (255, 255, 255), 10)
            clips.append(ImageClip(img).set_duration(0.7))

        # انتقال بعد التهجئة
        black = np.zeros((size[1], size[0], 3), dtype=np.uint8)
        clips.append(ImageClip(black).set_duration(0.3))

    # ── إزالة آخر انتقال ──
    if clips and clips[-1].duration <= 0.3:
        clips = clips[:-1]

    # ── دمج ──
    if clips:
        final = concatenate_videoclips(clips, method='compose')
        out = os.path.join(FINAL_DIR, 'sign_output.mp4')
        final.write_videofile(out, fps=24, codec='libx264',
                              audio=False, logger=None)

        # عرض الفيديو في Notebook
        with open(out, 'rb') as f:
            mp4 = b64encode(f.read()).decode()
        display(HTML(
            f'<div style="text-align:center; margin:20px; padding:15px; '
            f'background:#111; border-radius:15px;">'
            f'<h3 style="color:#4CAF50;">🎬 Sign Language Video</h3>'
            f'<p style="color:#aaa;">Duration: {final.duration:.1f}s | '
            f'{len(found)} signs + {len(not_found)} spelled</p>'
            f'<video width="640" controls autoplay style="border-radius:10px;">'
            f'<source src="data:video/mp4;base64,{mp4}" type="video/mp4">'
            f'</video></div>'
        ))
        return out
    return None


# ══════════════════════════════════════════════════════════════
#  🚀 الـ Pipeline الرئيسي
# ══════════════════════════════════════════════════════════════

def pipeline_v2(audio_path):
    """صوت → نص → معالجة → فيديو إشارة متتالي"""

    display(HTML(
        '<div style="text-align:center; padding:12px; margin:10px 0; '
        'background:linear-gradient(90deg,#1a73e8,#34a853); '
        'border-radius:12px; color:white; font-size:20px; font-weight:bold;">'
        '🚀 Speech → Sign Language Pipeline</div>'
    ))

    # ── 1. تحويل الصوت إلى نص ──
    print("🎤 [1/3] Transcribing...")
    heard = transcribe_audio(audio_path)
    if not heard:
        print("❌ No speech detected!")
        return

    display(HTML(
        f'<div style="padding:12px; margin:10px 0; background:#e8f5e9; '
        f'border-radius:10px; border-left:4px solid #4CAF50;">'
        f'<b>🎤 Heard:</b> <span style="font-size:18px;">"{heard}"</span></div>'
    ))

    # ── 2. مطابقة الكلمات ──
    print("🔍 [2/3] Matching words...")
    found, skipped, not_found = match_words(heard)

    display(HTML(
        f'<div style="padding:12px; margin:10px 0; background:#fff3e0; '
        f'border-radius:10px; border-left:4px solid #FF9800;">'
        f'<b>📊 Results:</b><br>'
        f'✅ Signs: <b>{found}</b><br>'
        f'🔤 Fingerspell: <b>{not_found}</b><br>'
        f'⏭️ Skipped: {skipped[:5]}</div>'
    ))

    # ── 3. إنشاء الفيديو المتتالي ──
    print(f"🎬 [3/3] Creating sign video...")
    if found or not_found:
        return create_sign_video(found, not_found)
    else:
        print("❌ No words to translate!")
        return None


# ══════════════════════════════════════════════════════════════
#  تشغيل الـ Pipeline
# ══════════════════════════════════════════════════════════════

audio_file = None
for path in ['/content/recorded_speech.wav',
             '/content/recorded_speech.webm',
             '/content/recorded_audio.wav']:
    if os.path.exists(path):
        audio_file = path
        size = os.path.getsize(path) / 1024
        print(f"📁 Found audio: {path} ({size:.1f} KB)")
        break

if audio_file:
    result = pipeline_v2(audio_file)
    if result:
        print(f"\n✅ Done! Video saved: {result}")
else:
    print("❌ No audio found! Record first, then run this cell.")

✅ Whisper: openai-whisper
✅ Video cache: 100/100 words
📁 Found audio: /content/recorded_speech.wav (120.1 KB)


🎤 [1/3] Transcribing...


🔍 [2/3] Matching words...


🎬 [3/3] Creating sign video...



✅ Done! Video saved: /content/drive/MyDrive/WLASL_Project/final_videos/sign_output.mp4


In [70]:
# ================================================================
#  🌐 Gradio Web Interface - Speech to Sign Language
# ================================================================

# تثبيت Gradio
!pip install -q gradio

import gradio as gr
import os, json, difflib, cv2, numpy as np
from moviepy.editor import VideoFileClip, ImageClip, concatenate_videoclips

# ── المسارات ──
VIDEOS_DIR = '/content/drive/MyDrive/wlasl_data/videos'
WLASL_JSON = '/content/drive/MyDrive/wlasl_data/WLASL_v0.3.json'
CLASS_JSON = '/content/drive/MyDrive/WLASL_Project/class_indices.json'
SIGN_DIR = '/content/drive/MyDrive/WLASL_Project/sign_videos'
FINAL_DIR = '/content/drive/MyDrive/WLASL_Project/final_videos'
os.makedirs(SIGN_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

# ── تحميل البيانات ──
with open(CLASS_JSON, 'r') as f:
    class_indices = json.load(f)
with open(WLASL_JSON, 'r') as f:
    wlasl_data = json.load(f)

available_words = list(class_indices.keys())

# بناء word_video_cache
word_video_cache = {}
for entry in wlasl_data:
    word = entry['gloss'].lower()
    if word in set(available_words) and word not in word_video_cache:
        for instance in entry['instances']:
            vid_id = instance['video_id']
            vid_path = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
            if os.path.exists(vid_path):
                cap = cv2.VideoCapture(vid_path)
                frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                cap.release()
                if frames > 0:
                    word_video_cache[word] = vid_id
                    break

STOP_WORDS = {
    'a', 'an', 'the', 'is', 'am', 'are', 'was', 'were', 'be',
    'to', 'of', 'in', 'for', 'on', 'at', 'by', 'with', 'from',
    'and', 'or', 'if', 'so', 'not', 'do',
    'my', 'your', 'his', 'her', 'its', 'our', 'their',
    'this', 'that', 'it', 'i', 'me', 'you', 'he', 'she',
    'we', 'they', 'have', 'has', 'had', 'will', 'would',
    'could', 'should', 'may', 'might', 'shall'
}

lemma_map = {
    'walked': 'walk', 'walking': 'walk', 'walks': 'walk',
    'ate': 'eat', 'eating': 'eat', 'eats': 'eat',
    'drank': 'drink', 'drinking': 'drink', 'drinks': 'drink',
    'played': 'play', 'playing': 'play', 'plays': 'play',
    'liked': 'like', 'liking': 'like', 'likes': 'like',
    'worked': 'work', 'working': 'work', 'works': 'work',
    'studied': 'study', 'studying': 'study', 'studies': 'study',
    'danced': 'dance', 'dancing': 'dance', 'dances': 'dance',
    'cooked': 'cook', 'cooking': 'cook', 'cooks': 'cook',
    'gave': 'give', 'giving': 'give', 'gives': 'give',
    'told': 'tell', 'telling': 'tell', 'tells': 'tell',
    'met': 'meet', 'meeting': 'meet', 'meets': 'meet',
    'needed': 'need', 'needing': 'need', 'needs': 'need',
    'enjoyed': 'enjoy', 'enjoying': 'enjoy', 'enjoys': 'enjoy',
    'forgot': 'forget', 'forgetting': 'forget', 'forgets': 'forget',
    'helped': 'help', 'helping': 'help', 'helps': 'help',
    'wanted': 'want', 'wanting': 'want', 'wants': 'want',
    'painted': 'paint', 'painting': 'paint', 'paints': 'paint',
    'changed': 'change', 'changing': 'change', 'changes': 'change',
    'decided': 'decide', 'deciding': 'decide', 'decides': 'decide',
    'finished': 'finish', 'finishing': 'finish', 'finishes': 'finish',
    'graduated': 'graduate', 'graduating': 'graduate',
    'pulled': 'pull', 'pulling': 'pull', 'pulls': 'pull',
}

# ── تحميل Whisper ──
try:
    from faster_whisper import WhisperModel
    wm = WhisperModel("base", device="auto", compute_type="int8")
    USE_FASTER = True
except:
    import whisper
    wm = whisper.load_model("base")
    USE_FASTER = False

# ══════════════════════════════════════════════════════════════
#  الدوال
# ══════════════════════════════════════════════════════════════

def transcribe(audio_path):
    if not audio_path:
        return ""
    try:
        if USE_FASTER:
            segments, _ = wm.transcribe(audio_path, language="en",
                                         beam_size=5, vad_filter=True)
            return " ".join(seg.text for seg in segments).strip().lower()
        else:
            result = wm.transcribe(audio_path, language="en")
            return result["text"].strip().lower()
    except:
        return ""

def process_text(text):
    if not text:
        return [], [], []

    found, skipped, not_found = [], [], []

    for word in text.split():
        clean = word.strip(".,!?;:'\"()[]").lower()

        if clean in lemma_map:
            clean = lemma_map[clean]

        if not clean or clean in STOP_WORDS or len(clean) <= 1:
            skipped.append(clean)
            continue

        if clean in available_words:
            found.append(clean)
        else:
            matches = difflib.get_close_matches(clean, available_words, n=1, cutoff=0.7)
            if matches:
                found.append(matches[0])
            else:
                not_found.append(clean)

    return found, skipped, not_found

def create_sign_video(found, not_found, size=(640, 360)):
    """
    فيديو إشارة متتالي - بدون كتابة الكلمات
    كل فيديو يتبعه الآخر مباشرة مع انتقال أسود قصير
    """
    clips = []

    # ── كلمات معروفة: فيديو إشارة فقط ──
    for word in found:
        vid_id = word_video_cache.get(word.lower())
        if vid_id:
            processed = os.path.join(SIGN_DIR, f'{word.lower()}.mp4')
            if not os.path.exists(processed):
                src = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
                os.system(
                    f'ffmpeg -y -i "{src}" -c:v libx264 -preset fast '
                    f'-crf 23 -an -vf "scale={size[0]}:{size[1]}" '
                    f'"{processed}" > /dev/null 2>&1'
                )
            if os.path.exists(processed):
                try:
                    clip = VideoFileClip(processed)
                    if clip.duration > 4.0:
                        clip = clip.subclip(0, 4.0)
                    clips.append(clip)
                except:
                    pass

        # انتقال أسود قصير (0.3 ثانية) بين الإشارات
        black = np.zeros((size[1], size[0], 3), dtype=np.uint8)
        clips.append(ImageClip(black).set_duration(0.3))

    # ── كلمات غير معروفة: تهجئة ──
    for word in not_found:
        for letter in word.upper():
            img = np.zeros((size[1], size[0], 3), dtype=np.uint8)
            font = cv2.FONT_HERSHEY_SIMPLEX
            ts = cv2.getTextSize(letter, font, 6.0, 10)[0]
            cv2.putText(img, letter,
                        ((size[0] - ts[0]) // 2, (size[1] + ts[1]) // 2),
                        font, 6.0, (255, 255, 255), 10)
            clips.append(ImageClip(img).set_duration(0.7))

        # انتقال بعد التهجئة
        black = np.zeros((size[1], size[0], 3), dtype=np.uint8)
        clips.append(ImageClip(black).set_duration(0.3))

    # ── إزالة آخر انتقال ──
    if clips and clips[-1].duration <= 0.3:
        clips = clips[:-1]

    # ── دمج ──
    if clips:
        final = concatenate_videoclips(clips, method='compose')
        out = os.path.join(FINAL_DIR, 'gradio_output.mp4')
        final.write_videofile(out, fps=24, codec='libx264',
                              audio=False, logger=None)
        return out
    return None


# ══════════════════════════════════════════════════════════════
#  واجهة Gradio
# ══════════════════════════════════════════════════════════════

def speech_to_sign(audio):
    """الدالة الرئيسية لـ Gradio"""
    if not audio:
        return "", "", None

    text = transcribe(audio)
    if not text:
        return "❌ No speech detected", "", None

    found, skipped, not_found = process_text(text)

    report = f"🎤 Heard: \"{text}\"\n\n"
    report += f"✅ Signs found ({len(found)}): {', '.join(found)}\n"
    if not_found:
        report += f"🔤 Fingerspelled ({len(not_found)}): {', '.join(not_found)}\n"
    if skipped:
        report += f"⏭️ Skipped: {', '.join(skipped[:5])}\n"

    content_words = len(text.split()) - len(skipped)
    rate = len(found) / content_words * 100 if content_words > 0 else 0
    report += f"\n📊 Translation rate: {rate:.0f}%"

    vocab = f"Available words ({len(available_words)}): {', '.join(sorted(available_words))}"

    video_path = None
    if found or not_found:
        video_path = create_sign_video(found, not_found)

    return report, vocab, video_path


# ── بناء الواجهة ──
with gr.Blocks(
    title="Speech to Sign Language",
    theme=gr.themes.Soft(primary_hue="blue")
) as app:

    gr.Markdown("""
    # 🤟 Speech to Sign Language Translator
    Record your voice or upload an audio file, and see the sign language translation!
    """)

    with gr.Row():
        with gr.Column(scale=1):
            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="🎙️ Record or Upload Audio"
            )
            translate_btn = gr.Button(
                "🚀 Translate to Sign Language",
                variant="primary",
                size="lg"
            )

            gr.Markdown("### 💡 Try these sentences:")
            examples = gr.Examples(
                examples=[
                    ["I want to eat pizza"],
                    ["help me drink"],
                    ["yes or no"],
                    ["I like to walk"],
                    ["book and computer"],
                    ["dark and hot"],
                    ["school play"],
                    ["dance and enjoy"],
                ],
                inputs=gr.Textbox(visible=False),
                label="Example sentences (use microphone)"
            )

        with gr.Column(scale=1):
            report_output = gr.Textbox(
                label="📊 Translation Report",
                lines=8
            )

    with gr.Row():
        video_output = gr.Video(
            label="🎬 Sign Language Video"
        )

    with gr.Accordion("📋 Available Vocabulary (100 words)", open=False):
        vocab_output = gr.Textbox(
            label="Words",
            lines=5
        )

    translate_btn.click(
        fn=speech_to_sign,
        inputs=audio_input,
        outputs=[report_output, vocab_output, video_output]
    )

app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://99ecfa744b64e6b53d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://99ecfa744b64e6b53d.gradio.live


In [72]:
# ================================================================
#  🌟 Gradio Professional Demo - لعرض لجنة التحكيم
# ================================================================

!pip install -q gradio

import gradio as gr
import os, json, difflib, cv2, numpy as np
from moviepy.editor import (VideoFileClip, ImageClip,
                             concatenate_videoclips, CompositeVideoClip,
                             TextClip)

# ── المسارات ──
VIDEOS_DIR = '/content/drive/MyDrive/wlasl_data/videos'
WLASL_JSON = '/content/drive/MyDrive/wlasl_data/WLASL_v0.3.json'
CLASS_JSON = '/content/drive/MyDrive/WLASL_Project/class_indices.json'
SIGN_DIR = '/content/drive/MyDrive/WLASL_Project/sign_videos'
FINAL_DIR = '/content/drive/MyDrive/WLASL_Project/final_videos'
os.makedirs(SIGN_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

# ── تحميل البيانات ──
with open(CLASS_JSON, 'r') as f:
    class_indices = json.load(f)
with open(WLASL_JSON, 'r') as f:
    wlasl_data = json.load(f)

available_words = list(class_indices.keys())

word_video_cache = {}
for entry in wlasl_data:
    word = entry['gloss'].lower()
    if word in set(available_words) and word not in word_video_cache:
        for instance in entry['instances']:
            vid_id = instance['video_id']
            vid_path = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
            if os.path.exists(vid_path):
                cap = cv2.VideoCapture(vid_path)
                frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                cap.release()
                if frames > 0:
                    word_video_cache[word] = vid_id
                    break

STOP_WORDS = {
    'a', 'an', 'the', 'is', 'am', 'are', 'was', 'were', 'be',
    'to', 'of', 'in', 'for', 'on', 'at', 'by', 'with', 'from',
    'and', 'or', 'if', 'so', 'not', 'do',
    'my', 'your', 'his', 'her', 'its', 'our', 'their',
    'this', 'that', 'it', 'i', 'me', 'you', 'he', 'she',
    'we', 'they', 'have', 'has', 'had', 'will', 'would',
    'could', 'should', 'may', 'might', 'shall'
}

lemma_map = {
    'walked': 'walk', 'walking': 'walk', 'walks': 'walk',
    'ate': 'eat', 'eating': 'eat', 'eats': 'eat',
    'drank': 'drink', 'drinking': 'drink', 'drinks': 'drink',
    'played': 'play', 'playing': 'play', 'plays': 'play',
    'liked': 'like', 'liking': 'like', 'likes': 'like',
    'worked': 'work', 'working': 'work', 'works': 'work',
    'studied': 'study', 'studying': 'study', 'studies': 'study',
    'danced': 'dance', 'dancing': 'dance', 'dances': 'dance',
    'cooked': 'cook', 'cooking': 'cook', 'cooks': 'cook',
    'gave': 'give', 'giving': 'give', 'gives': 'give',
    'told': 'tell', 'telling': 'tell', 'tells': 'tell',
    'met': 'meet', 'meeting': 'meet', 'meets': 'meet',
    'needed': 'need', 'needing': 'need', 'needs': 'need',
    'enjoyed': 'enjoy', 'enjoying': 'enjoy', 'enjoys': 'enjoy',
    'forgot': 'forget', 'forgetting': 'forget', 'forgets': 'forget',
    'helped': 'help', 'helping': 'help', 'helps': 'help',
    'wanted': 'want', 'wanting': 'want', 'wants': 'want',
    'painted': 'paint', 'painting': 'paint', 'paints': 'paint',
    'changed': 'change', 'changing': 'change', 'changes': 'change',
    'decided': 'decide', 'deciding': 'decide', 'decides': 'decide',
    'finished': 'finish', 'finishing': 'finish', 'finishes': 'finish',
    'graduated': 'graduate', 'graduating': 'graduate',
    'pulled': 'pull', 'pulling': 'pull', 'pulls': 'pull',
}

# ── تحميل Whisper ──
try:
    from faster_whisper import WhisperModel
    wm = WhisperModel("base", device="auto", compute_type="int8")
    USE_FASTER = True
except:
    import whisper
    wm = whisper.load_model("base")
    USE_FASTER = False

# ══════════════════════════════════════════════════════════════
#  الدوال
# ══════════════════════════════════════════════════════════════

def transcribe(audio_path):
    if not audio_path:
        return ""
    try:
        if USE_FASTER:
            segments, _ = wm.transcribe(audio_path, language="en",
                                         beam_size=5, vad_filter=True)
            return " ".join(seg.text for seg in segments).strip().lower()
        else:
            result = wm.transcribe(audio_path, language="en")
            return result["text"].strip().lower()
    except:
        return ""


def process_text(text):
    if not text:
        return [], [], []
    found, skipped, not_found = [], [], []
    for word in text.split():
        clean = word.strip(".,!?;:'\"()[]").lower()
        if clean in lemma_map:
            clean = lemma_map[clean]
        if not clean or clean in STOP_WORDS or len(clean) <= 1:
            skipped.append(clean)
            continue
        if clean in available_words:
            found.append(clean)
        else:
            matches = difflib.get_close_matches(clean, available_words, n=1, cutoff=0.7)
            if matches:
                found.append(matches[0])
            else:
                not_found.append(clean)
    return found, skipped, not_found


def create_bar_with_word(word, size=(640, 50)):
    """شريط سفلي يعرض الكلمة الحالية"""
    bar = np.zeros((size[1], size[0], 3), dtype=np.uint8)
    # خلفية داكنة
    bar[:, :] = [30, 30, 50]
    # اسم الكلمة
    font = cv2.FONT_HERSHEY_SIMPLEX
    ts = cv2.getTextSize(word.upper(), font, 1.0, 2)[0]
    cv2.putText(bar, word.upper(),
                ((size[0] - ts[0]) // 2, (size[1] + ts[1]) // 2),
                font, 1.0, (100, 220, 255), 2)
    # خط علوي
    cv2.line(bar, (0, 0), (size[0], 0), (100, 220, 255), 2)
    return bar


def create_sign_video(found, not_found, heard_text="", size=(640, 360)):
    """
    فيديو إشارة احترافي مع شريط يعرض الكلمة الحالية
    """
    clips = []
    video_h = size[1] - 50  # مساحة الفيديو
    bar_h = 50              # مساحة الشريط

    for word in found:
        vid_id = word_video_cache.get(word.lower())
        if vid_id:
            processed = os.path.join(SIGN_DIR, f'{word.lower()}.mp4')
            if not os.path.exists(processed):
                src = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
                os.system(
                    f'ffmpeg -y -i "{src}" -c:v libx264 -preset fast '
                    f'-crf 23 -an -vf "scale={size[0]}:{video_h}" '
                    f'"{processed}" > /dev/null 2>&1'
                )
            if os.path.exists(processed):
                try:
                    clip = VideoFileClip(processed)
                    if clip.duration > 4.0:
                        clip = clip.subclip(0, 4.0)

                    # تغيير الحجم ليناسب مساحة الفيديو
                    clip = clip.resize(newsize=(size[0], video_h))

                    # إنشاء شريط الكلمة
                    bar_img = create_bar_with_word(word, size=(size[0], bar_h))
                    bar_clip = ImageClip(bar_img).set_duration(clip.duration)

                    # دمج الفيديو + الشريط
                    composite = CompositeVideoClip([
                        clip.set_position(('center', 0)),
                        bar_clip.set_position(('center', video_h))
                    ], size=size)
                    composite = composite.set_duration(clip.duration)

                    clips.append(composite)
                except:
                    pass

        # انتقال أسود قصير
        black = np.zeros((size[1], size[0], 3), dtype=np.uint8)
        clips.append(ImageClip(black).set_duration(0.3))

    # ── تهجئة الكلمات غير المعروفة ──
    for word in not_found:
        for letter in word.upper():
            img = np.zeros((size[1], size[0], 3), dtype=np.uint8)
            font = cv2.FONT_HERSHEY_SIMPLEX
            ts = cv2.getTextSize(letter, font, 5.0, 8)[0]
            cv2.putText(img, letter,
                        ((size[0] - ts[0]) // 2, (size[1] - 50 + ts[1]) // 2),
                        font, 5.0, (255, 255, 255), 8)

            # شريط تهجئة
            bar_img = create_bar_with_word(f"Spell: {word}", size=(size[0], bar_h))
            img[video_h:, :] = bar_img

            clips.append(ImageClip(img).set_duration(0.7))

        black = np.zeros((size[1], size[0], 3), dtype=np.uint8)
        clips.append(ImageClip(black).set_duration(0.3))

    # إزالة آخر انتقال
    if clips and clips[-1].duration <= 0.3:
        clips = clips[:-1]

    if clips:
        final = concatenate_videoclips(clips, method='compose')
        out = os.path.join(FINAL_DIR, 'demo_output.mp4')
        final.write_videofile(out, fps=24, codec='libx264',
                              audio=False, logger=None)
        return out
    return None


# ══════════════════════════════════════════════════════════════
#  واجهة Gradio الاحترافية
# ══════════════════════════════════════════════════════════════

def speech_to_sign(audio):
    if not audio:
        return "", "", None

    text = transcribe(audio)
    if not text:
        return "❌ No speech detected", "", None

    found, skipped, not_found = process_text(text)

    # تقرير مفصل
    content_words = len(text.split()) - len(skipped)
    rate = len(found) / content_words * 100 if content_words > 0 else 0

    report = f"🎤 Heard: \"{text}\"\n\n"
    report += f"✅ Signs found ({len(found)}): {', '.join(found)}\n"
    if not_found:
        report += f"🔤 Fingerspelled ({len(not_found)}): {', '.join(not_found)}\n"
    if skipped:
        report += f"⏭️ Skipped: {', '.join(skipped[:5])}\n"
    report += f"\n📊 Translation rate: {rate:.0f}%\n"
    report += f"📚 Vocabulary: {len(available_words)} words\n"
    report += f"🤖 Model: MobileNetV2 (71.56% accuracy)"

    vocab = ", ".join(sorted(available_words))

    video_path = None
    if found or not_found:
        video_path = create_sign_video(found, not_found, heard_text=text)

    return report, vocab, video_path


with gr.Blocks(
    title="Speech to Sign Language Translator",
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="green"),
    css="""
    .main-title {text-align: center; margin-bottom: 5px;}
    .subtitle {text-align: center; color: #666; margin-bottom: 20px;}
    """
) as app:

    gr.HTML("""
    <div style="text-align:center; padding:20px; background:linear-gradient(135deg, #1a73e8, #34a853);
                border-radius:15px; color:white; margin-bottom:20px;">
        <h1 style="margin:0; font-size:28px;">🤟 Speech to Sign Language Translator</h1>
        <p style="margin:5px 0 0; font-size:16px; opacity:0.9;">
            Convert spoken English to American Sign Language (ASL) videos
        </p>
        <p style="margin:5px 0 0; font-size:13px; opacity:0.7;">
            MobileNetV2 • 100 Words • WLASL Dataset • 71.56% Accuracy
        </p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 🎙️ Input")
            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="filepath",
                label="Record or Upload Audio"
            )
            translate_btn = gr.Button(
                "🚀 Translate to Sign Language",
                variant="primary",
                size="lg"
            )

            gr.Markdown("### 💡 Example Sentences")
            gr.Markdown("""
            - *"I want to eat pizza"*
            - *"help me drink"*
            - *"yes or no"*
            - *"I like to walk"*
            - *"book and computer"*
            - *"dark and hot"*
            - *"school play"*
            - *"dance and enjoy"*
            """)

        with gr.Column(scale=1):
            gr.Markdown("### 📊 Translation Report")
            report_output = gr.Textbox(
                label="",
                lines=9
            )

    with gr.Row():
        video_output = gr.Video(
            label="🎬 Sign Language Translation Video"
        )

    with gr.Accordion("📋 Available Vocabulary (100 words)", open=False):
        vocab_output = gr.Textbox(label="", lines=4)

    with gr.Accordion("ℹ️ About the Project", open=False):
        gr.Markdown("""
        ### Speech to Sign Language Translator

        **Pipeline:**
        1. 🎤 Speech → Text (faster-whisper)
        2. 📝 Text → NLP Processing (lemmatization + stop words removal)
        3. 🔍 Word Matching (exact + fuzzy matching)
        4. 🎬 Sign Video Generation (WLASL dataset)
        5. 🔤 Fingerspelling for unknown words

        **Model:** MobileNetV2 fine-tuned on WLASL dataset
        - **Accuracy:** 71.56%
        - **Vocabulary:** 100 ASL signs
        - **Input:** 128x128 RGB images
        - **Dataset:** WLASL (Worcester Polytechnic Institute)

        **Technologies:** TensorFlow, faster-whisper, OpenCV, MoviePy, Gradio
        """)

    translate_btn.click(
        fn=speech_to_sign,
        inputs=audio_input,
        outputs=[report_output, vocab_output, video_output]
    )

app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b1b5bdbe3426eb636b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://b1b5bdbe3426eb636b.gradio.live
